<a href="https://colab.research.google.com/github/fighting-mochi/CV_public/blob/main/tyra_job_sniffer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#教育部大專教師人才網: https://tjn.moe.edu.tw/EduJin/Opening/Index; rss: https://www.nstc.gov.tw/nstc/rss/hr
#國家科技與技術委員會求才消息: https://www.nstc.gov.tw/folksonomy/list/ba3d22f3-96fd-4adf-a078-91a05b8f0166?l=ch

#https://urlwatch.readthedocs.io/en/latest/introduction.html#how-it-works
#https://medium.com/@aleksej.gudkov/python-script-to-monitor-website-changes-a-practical-guide-5387bef5056a

In [1]:
import os
import io
import subprocess
import re
import time
import datetime

import requests
from bs4 import BeautifulSoup
import lxml

from google.colab import auth, drive
from google.auth import default
import gspread
import json

import pandas as pd
import numpy as np

import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.image import MIMEImage
from PIL import Image

drive.mount('/content/drive/')

# Authenticate and authorize
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)


Mounted at /content/drive/


In [ ]:
# get data from rss feed of 國科會徵才資訊
target_url = "https://www.nstc.gov.tw/nstc/rss/hr"
filter = "助理教授" # 可能也要找"教授" "專案教授" "正教授"...

response = requests.get(target_url)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'xml')

links = soup.find_all('link')
titles = soup.find_all('title')
descriptions = soup.find_all('description')

job_links = []
job_titles = []
job_descriptions = []

year = datetime.datetime.now().year

for i, j, k in zip(titles[2:], links[2:], descriptions[2:]):
  raw_title = i.text
  job_title = raw_title[:-11]
  job_link = j.text
  job_description = k.text
  job_titles.append(job_title)
  job_links.append(job_link)
  job_descriptions.append(job_description)

df = pd.DataFrame({'title': job_titles, 'link': job_links, 'description': job_descriptions})


# save current run to csv
path_to_save = '/content/drive/MyDrive/job_sniffer_data/'
date = datetime.datetime.now().strftime("%Y%m%d")
file_to_save = 'job_sniffer_' + date + '.csv'

df_links = df

df_links['temp'] = df['title'] + df['description']

df_links = df[df['temp'].str.contains(filter, na=False)]

df_links = df_links.drop(['temp', 'description'], axis = 1)

df_links['date_added'] = date
'''
df_links.to_csv(file_to_save)

subprocess.run(["cp", file_to_save, path_to_save])

job_sniffer_sheet = gc.open_by_url("https://docs.google.com/spreadsheets/d/1rkJ2grICnS8Nm1ggZ80ga5Oeoujxm3mmm_4ujjm-6J0/").get_worksheet(0)

# add to google sheets above
job_sniffer_sheet.append_rows(df_links.values.tolist())

# clean up duplicates
data = job_sniffer_sheet.get_all_values()

# Remove duplicates using a set
unique_rows = []
seen_rows = set()
for row in data:
    row_tuple = tuple(row) # Convert list to tuple to make it hashable for the set
    if row_tuple not in seen_rows:
        unique_rows.append(row)
        seen_rows.add(row_tuple)

# Clear the existing content and update with unique rows
job_sniffer_sheet.clear()
job_sniffer_sheet.update(unique_rows)


# add hyperlinks to job titles
updates = []
for i, text in enumerate(job_sniffer_sheet.col_values(1)):
    if i == 0:
        continue

    target_cell_ref = f"B{i+1}"

    # Construct the HYPERLINK formula
    hyperlink_formula = f'=HYPERLINK({target_cell_ref}, "{text}")'
    updates.append([hyperlink_formula])

job_sniffer_sheet.update(updates, 'A2', value_input_option='USER_ENTERED')

# with open(path_to_save + 'file_tracker.txt', 'w') as f:
#   f.write(file_to_save)

# then next time save to a csv with a different name
# and compare 2 sheets when running next time, identify and show new jobs, and pick which jobs to send
'''
df_links


In [ ]:
df_links.iloc[1]['title']

'國立台灣海洋大學【永續發展碩士在職學位學程(所)】新聘「「永續發展」領域專長，具環境（科學、工程、教育、規劃與政策）、生態保育、景觀、地理、永續發展及文化資產保存等專長」助理教授(含)以上專任教師1名公告'

archive below

In [ ]:
current_run_filepath = '/content/drive/MyDrive/job_sniffer_data/' + file_to_save

previous_run_filename = 'job_sniffer_20250610.csv' #open('/content/drive/MyDrive/job_sniffer_data/file_tracker.txt', 'r').read()

previous_run_filepath = '/content/drive/MyDrive/job_sniffer_data/' + previous_run_filename

if file_to_save == previous_run_filename:
  print("no new data")
elif not os.path.exists(previous_run_filepath):
  print(f"Error: File '{previous_run_filepath}' not found.")
else:
  print("current run: ", current_run_filepath)
  print("comparing with previous run: ", previous_run_filepath)

  df_current = pd.read_csv(current_run_filepath)
  df_previous = pd.read_csv(previous_run_filepath)

  df_merged = pd.merge(df_current, df_previous, how="outer")

  df_new_entries = df_merged.drop_duplicates(subset=['link'], keep=False)
  df_new_entries = df_new_entries.drop(columns=['Unnamed: 0'])

  print("updated dataframe:")

df_new_entries

# maybe add some filter for > prof
# make some table to consolidate things for all the jobs (maybe weekly email and then put all jobs in a table)
# think about how to include jobs from different sources (i.e. academic institutions in taiwan, start from 4大, 中研院 etc), beyond 國科會, format all into some table into some email

current run:  /content/drive/MyDrive/job_sniffer_data/job_sniffer_20250810.csv
comparing with previous run:  /content/drive/MyDrive/job_sniffer_data/job_sniffer_20250610.csv
new entries:


,title,link,description
0,國立臺灣大學流行病學與預防醫學研究所誠徵專任教師1名,https://www.nstc.gov.tw/folksonomy/detail/9f6a...,<p>徵求職級: 專任助理教授/副教授\n<br>\n<br>一、工作概述\n<br>國立臺...
1,【徵才】國立中央大學研究核心設施中心誠徵 雙束型聚焦離子束系統 技術人員,https://www.nstc.gov.tw/folksonomy/detail/613a...,<p>【徵才】國立中央大學研究核心設施中心誠徵 雙束型聚焦離子束系統 技術人員\n<br>【...
2,國立高雄科技大學供應鏈管理系徵聘專任教師公告(延長公告),https://www.nstc.gov.tw/folksonomy/detail/17c1...,NaN


In [ ]:
# Gmail SMTP configuration
smtp_server = 'smtp.gmail.com'
smtp_port = 587
sender_email = 'public@projecttyra.org'
sender_password = 'wjrc hzul sjgw kysr'
recipient_email = 'tyra-jobs-opp@projecttyra.org'

data_to_send = df_new_entries.iloc[[1]]

icon_path = "/content/drive/MyDrive/job_sniffer_data/robot.png"

# process jobs in the data_to_send dataframe
for index, row in data_to_send.iterrows():

  job_title = row['title']
  job_link = row['link']
  job_description = row['description']

  print("processing job: ", job_title)

  message = MIMEMultipart()
  message['From'] = sender_email
  message['To'] = recipient_email
  message['Subject'] = job_title

  body = f"<p>Hi Project TYRA 的朋友們,</p>"\
         f"<p>感謝 tyra_job_sniffer <img src='cid:logo'> 以及國科會幫忙提供以下工作資訊</p>"\
         f"<p> </p>"\
         f"<p>=====</p>"\
         f"<p><b>{job_title}</b><a href={job_link}> (國科會原始網站連結)</a></p>"\
         f"<p>{job_description}</p>"\
         f"<p> </p>"\
         f"<p>=====</p>"\
         f"<p>Project TYRA 祝您求職順利</p>"\

  message.attach(MIMEText(body, 'html'))

  # Open and resize the image
  with Image.open(icon_path) as image:
    width, height = image.size
    new_width = int(width * 0.05)  # Reduce the width by 50%
    new_height = int(height * 0.05)  # Reduce the height by 50%" \
    resized_image = image.resize((new_width, new_height))

  # Convert the resized logo image to bytes
  with io.BytesIO() as output:
    resized_image.save(output, format='PNG')
    logo_data = output.getvalue()

  # Attach the logo image to the email
    logo = MIMEImage(logo_data)
    logo.add_header('Content-ID', '<logo>')
    logo.add_header('Content-Disposition', 'inline', filename='robot.png')
    message.attach(MIMEText(body, 'html'))
    message.attach(logo)

  # send the message to tyra-jobs-opp for approval
  with smtplib.SMTP(smtp_server, smtp_port) as server:
        server.starttls()
        server.login(sender_email, sender_password)
        server.send_message(message)
        print(f"Email sent to {recipient_email} for {job_title} successfully.")

  time.sleep(2)


processing job:  國立臺灣大學流行病學與預防醫學研究所誠徵專任教師1名
Email sent to tyra-jobs-opp@projecttyra.org for 國立臺灣大學流行病學與預防醫學研究所誠徵專任教師1名 successfully.


In [ ]:
#print(len(job_descriptions))

# for i in job_descriptions[0:2]:
#   print("=====")
#   print(i)
#   with open('test.txt', "a") as file:
#     file.write("=====")
#     file.write(str(i))

In [ ]:
# test_post = df.head(1)

# test_post_link = test_post['link'][0]
# test_post_title = test_post['title'][0]

# #print(test_post_title)

# job_link_page = requests.get(test_post_link)
# job_link_page.raise_for_status()

# soup_job = BeautifulSoup(job_link_page.text, 'lxml').get_text()

# print(soup_job)

#soup_job_text = str(soup_job)

#string = '<h2 class="articleTitle" name="start-content">'

#soup_job_text[soup_job_text.find(string):]

#print(soup_job_text)

#test = soup_job.find('h2 class="articleTitle" name="start-content"')

In [ ]:
#page_content = soup_job.find(string=test_post_title)
#print(page_content)

#print(soup_job.prettify())

#   print("=====")
#   job_link_page = requests.get(i)
#   job_link_page.raise_for_status()

#   soup_job = BeautifulSoup(job_link_page.text, 'html.parser')
#   print(soup_job.prettify())


# print(job_links)

# # initialize the list of discovered URLs
# urls_to_visit = [target_url]

# # set a maximum crawl limit
# max_crawl = 20

# def crawler():
#     # set a crawl counter to track the crawl depth
#     crawl_count = 0

#     while urls_to_visit and crawl_count < max_crawl:

#         # get the page to visit from the list
#         current_url = urls_to_visit.pop()

#         # request the target URL
#         response = requests.get(current_url)
#         response.raise_for_status()
#         # parse the HTML
#         soup = BeautifulSoup(response.text, "html.parser")

#         # collect all the links
#         link_elements = soup.select("a[href]")
#         for link_element in link_elements:
#             url = link_element["href"]

#             # convert links to absolute URLs
#             if not url.startswith("http"):
#                 absolute_url = requests.compat.urljoin(target_url, url)
#             else:
#                 absolute_url = url

#             # ensure the crawled link belongs to the target domain and hasn't been visited
#             if (
#                 absolute_url.startswith(target_url)
#                 and absolute_url not in urls_to_visit
#             ):
#                 urls_to_visit.append(absolute_url)

#             # update the crawl count
#             crawl_count += 1

#     # print the crawled URLs
#     print(urls_to_visit)

# # execute the crawl
# crawler()

# Lan-Tien's attempt

In [2]:
import os
import io
import subprocess
import re
import time
import datetime

import requests
from bs4 import BeautifulSoup
import lxml

from google.colab import auth, drive
from google.auth import default
import gspread
import json

import pandas as pd
import numpy as np
from urllib.parse import urljoin


import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.image import MIMEImage
from PIL import Image

drive.mount('/content/drive/')

# Authenticate and authorize
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)


Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [83]:
def extract_data(soup: BeautifulSoup):
    def parse_date(date_str: str):
        return datetime.datetime.strptime(date_str, '%Y/%m/%d')

    def extract_datum(r: BeautifulSoup):
        institution, position, area, announcement_date, deadline_date, description = [ td.text.strip() for td in r.find_all('td') ]
        url = urljoin(url_base, r.a['href'])
        # institution, position, area, announcement_date, deadline_date, description = [ td.text.strip() for td in r.find_all('td') ]
        # url = urljoin(url_base, r.a['href'])

        opening = {
            'institution': institution,
            'position': position,
            'area': area,
            'announcement_date': parse_date(announcement_date),
            'deadline_date': parse_date(deadline_date),
            'description_number': description,
            'url': url,
        }

        return opening

    res = soup.find('table', id='SearchTable').find_all('tr', class_='listColor')

    return [ extract_datum(r) for r in res ]

In [109]:
import urllib3
urllib3.disable_warnings()

# change here
PostDateStart = "2026-03-24"
PostDateEnd = "2026-04-25"

# no need to change
today = datetime.datetime.now().strftime(format='%Y-%m-%d')
url_base = 'https://tjn.moe.edu.tw'
url_post = 'https://tjn.moe.edu.tw/EduJin/Opening/Search'
url_head = 'https://tjn.moe.edu.tw/EduJin/Opening/Detail?num='

# session = requests.Session()
# session.verify = False

page = 1
df = pd.DataFrame({})
df_hide = pd.DataFrame({})


while True:
# while page == 14:
  response = requests.post(url_post,
      data = { "PostDateStart": PostDateStart, "PostDateEnd": PostDateEnd, "Page":page}, verify=False
  )
  # response = session.get(url_post, params={ 'page': page })

  soup = BeautifulSoup(response.text, 'html.parser')
  res = soup.find_all("td", class_="md-hide") # from md-hide

  # res = soup.find('table', id='SearchTable').find_all('tr', class_='listColor') # in the function: extract_data(soup)
  df_new = pd.DataFrame(extract_data(soup))

  if (len(df_new) != 0) and (len(res) != 0):
    df_new['description'] = df_new['description_number'].str.replace(r'\n+\d+\s*$', '', regex=True)
    df = pd.concat([df, df_new], ignore_index=True)
    # 'position': '新生醫護管理專科學校115學年度第1學期通識中心全民國防教育兼任教師徵才公告',
    # 'area': '桃園市龍潭區',
    # 'announcement_date': datetime.datetime(2026, 4, 15, 0, 0),
    # 'deadline_date': datetime.datetime(2026, 7, 31, 0, 0),
    # 'url': 'https://tjn.moe.edu.tw/EduJin/Opening/Detail?num=29346'}]

    for r in res:
      # chunk = r.find('div')
      onclick = r.div['onclick']
      link = url_base + re.search(r"'(.*)'", onclick)[1]
      description = r.find_all('span')[1].get_text()
      location = r.find_all('span')[2].get_text()
      department = re.sub(r'^.*?大學', '', r.find_all('span')[0].get_text())
      # print(description, link)
      df_hide_new = pd.DataFrame([[department, description, link, location, today]], columns=['department', 'description', 'link', 'location', 'extracted date'])
      df_hide = pd.concat([df_hide, df_hide_new], ignore_index=True)
      # print(f'{len(df_hide_new)} lines on page {page} is saved.')


  elif (len(df_new) == 0) and (len(res) != 0):
    print('something went wrong...')
    print(len(df_new), len(res))
    break
  elif (len(df_new) != 0) and (len(res) == 0):
    print('something went wrong...')
    print(len(df_new), len(res))
    break
  elif (len(df_new) == 0) and (len(res) == 0):
    print('no new data')
    break
  page += 1
  print('page:', page)

if len(df) != len(df_hide):
  print('something went wrong....')
else:
  result = pd.merge(df, df_hide, on="description")

df = result
# clean up duplicates
df = df.drop_duplicates('description')

# Convert datetime columns to string before saving to Google Sheet
datetime_columns = ['announcement_date', 'deadline_date']
for col in datetime_columns:
    if col in df.columns:
        df[col] = df[col].dt.strftime('%Y-%m-%d')

# add hyperlink
df["hyperlink"] = df.apply(
    lambda row: f'=HYPERLINK("{row["link"]}", "{row["description"]}")', # Corrected the f-string syntax
    axis=1
)

df = df.sort_values(by='department', ascending=False)

# save to google worksheet (copy to tyra newsletter)
job_sniffer_sheet = gc.open_by_key("1rkJ2grICnS8Nm1ggZ80ga5Oeoujxm3mmm_4ujjm-6J0").worksheet('EduJin')
job_sniffer_sheet.append_rows(df.values.tolist())

# save as csv file
csvfile = f'/content/drive/Shareddrives/tyra_workspace/6) IT Group/3) Projects/JobMarket/job_sniffer_data/job_sniffer_{today}.csv'
df.to_csv(csvfile, encoding='utf-8', index=False)

page: 2
page: 3
page: 4
page: 5
page: 6
page: 7
page: 8
page: 9
page: 10
page: 11
page: 12
page: 13
page: 14
no new data


/tmp/ipykernel_2274/2674510034.py:86: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].dt.strftime('%Y-%m-%d')
/tmp/ipykernel_2274/2674510034.py:86: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].dt.strftime('%Y-%m-%d')
/tmp/ipykernel_2274/2674510034.py:89: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-do

In [110]:
# export to the public

spreadsheet_key = "1zFIVts9qAlKWJbIQzhUaWEdza6x6eT2UvH-kINO1AoE"
worksheet_name = today
headers = [['department', 'hyperlink']] # in principle, i have also 'announced date, deadline_date, location...', but i didn't export it.

try:
    spreadsheet = gc.open_by_key(spreadsheet_key)
    current_data_start_row = 1 # Default, if no existing data/headers

    try:
        job_sniffer_sheet = spreadsheet.worksheet(worksheet_name)
        print(f"Worksheet '{worksheet_name}' already exists.")
        all_existing_values = job_sniffer_sheet.get_all_values()

        if not all_existing_values: # If the sheet is truly empty
            job_sniffer_sheet.update(headers, 'A1', value_input_option='RAW')
            # job_sniffer_sheet.update([df.columns.values.tolist()], 'A1', value_input_option='RAW')
            current_data_start_row = 2 # Data starts from the second row after headers
        else: # Sheet has existing content (could be headers or headers + data)
            # Calculate the next available row for new data
            current_data_start_row = len(all_existing_values) + 1

    except gspread.exceptions.WorksheetNotFound:
        # Create the new worksheet
        job_sniffer_sheet = spreadsheet.add_worksheet(title=worksheet_name, rows=len(df), cols=2)
        print(f"Worksheet '{worksheet_name}' created successfully.")
        job_sniffer_sheet.update(headers, 'A1', value_input_option='RAW')
        # job_sniffer_sheet.update([df.columns.values.tolist()], f'A1', value_input_option='RAW')
        current_data_start_row = 2

    if not df.empty:
        # Prepare a list of lists for the combined data (hyperlink, description, link, extracted date)
        # Ensure the order matches the desired column order in the sheet
        # Assuming the sheet columns are: hyperlink, description, link, extracted date
        combined_data = []
        for index, row in df.iterrows():
            combined_data.append([row['department'], row['hyperlink']])  # A: department, B: hyperlink [code_file:2]

        num_rows_to_update = len(combined_data)
        if num_rows_to_update == 0:
            print("DataFrame 'df' is empty, no data to append.")

        # Determine the full range for the update operation
        # Assuming 4 columns: A, B, C, D
        full_data_range = f'A{current_data_start_row}:B{current_data_start_row + num_rows_to_update - 1}'

        # Update the sheet with the data, interpreting formulas
        job_sniffer_sheet.update(combined_data, full_data_range, value_input_option='USER_ENTERED')
        print(f"Data from DataFrame 'df' appended to worksheet '{worksheet_name}' with hyperlinks.")
    else:
        print("DataFrame 'df' is empty, no data to append.")

except gspread.exceptions.SpreadsheetNotFound:
    print(f"Error: Spreadsheet with key '{spreadsheet_key}' not found. Please check the spreadsheet key.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Worksheet '2026-04-24' created successfully.
Data from DataFrame 'df' appended to worksheet '2026-04-24' with hyperlinks.


In [208]:
# another approach
from collections.abc import Iterable
from datetime import datetime
from itertools import chain, count, takewhile
from urllib.parse import urljoin
from bs4 import BeautifulSoup

import polars as pl
import requests
import urllib3

urllib3.disable_warnings()


url_base = 'https://tjn.moe.edu.tw'
url_openings = 'https://tjn.moe.edu.tw/EduJin/Opening'


def extract_data(soup: BeautifulSoup):
    def parse_date(date_str: str) -> datetime:
        return datetime.strptime(date_str, '%Y/%m/%d')

    def extract_datum(r: BeautifulSoup):
        institution, position, area, announcement_date, deadline_date, _ = [ td.text.strip() for td in r.find_all('td') ]
        url = urljoin(url_base, r.a['href'])

        opening = {
            'institution': institution,
            'position': position,
            'area': area,
            'announcement_date': parse_date(announcement_date),
            'deadline_date': parse_date(deadline_date),
            'url': url
        }

        return opening

    res = soup.find('table', id='SearchTable').find_all('tr', class_='listColor')

    return [ extract_datum(r) for r in res ]


def get_page(session: requests.Session, page: int) -> BeautifulSoup:
    print('.')
    res = session.get(url_openings, params={ 'page': page })
    return BeautifulSoup(res.text, 'html.parser')


def get_all_openings(session: requests.Session):
    return chain.from_iterable(takewhile(lambda _: _, (extract_data(get_page(session, i)) for i in count(1))))


def existing_data() -> pl.DataFrame:
    return pl.read_parquet('all.parquet')


def write_data(df: pl.DataFrame) -> None:
    df.write_parquet('new.parquet')


session = requests.Session()
session.verify = False


new_df = pl.DataFrame(get_all_openings(session)).reverse()
print(new_df)
print(len(new_df))

.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
shape: (413, 6)
┌────────────────┬────────────────┬──────────────┬────────────────┬────────────────┬───────────────┐
│ institution    ┆ position       ┆ area         ┆ announcement_d ┆ deadline_date  ┆ url           │
│ ---            ┆ ---            ┆ ---          ┆ ate            ┆ ---            ┆ ---           │
│ str            ┆ str            ┆ str          ┆ ---            ┆ datetime[μs]   ┆ str           │
│                ┆                ┆              ┆ datetime[μs]   ┆                ┆               │
╞════════════════╪════════════════╪══════════════╪════════════════╪════════════════╪═══════════════╡
│ 國立成功大學航 ┆ 國立成功大學太 ┆ 臺南市東區   ┆ 2024-01-01     ┆ 2026-12-31     ┆ https://tjn.m │
│ 太系           ┆ 空系統工程研究 ┆              ┆ 00:00:00       ┆ 00:00:00       ┆ oe.edu.tw/Edu │
│                ┆ 所徵聘專任教師 ┆              ┆                ┆                ┆ Jin/…         │
│ 國立成功大學航 ┆ 國立成功大學航 ┆ 臺南市東區   ┆ 2024-01-01     ┆ 2026-12-31     ┆ 

In [215]:
csvfile = f'/content/drive/Shareddrives/tyra_workspace/6) IT Group/3) Projects/JobMarket/job_sniffer_data/job_sniffer_{today}_ALL_testing.csv'
new_df.write_csv(csvfile)